# SIGIR 2022 Showcase

- showcase some features of the [`ir_axioms`](https://github.com/webis-de/ir_axioms) framework
- example data:
  - MSMARCO documents and index
  - TREC Deep Learning topics


## Preparation


1. Download MSMARCO and TREC 2019/2020 Deep Learning tracks (using `ir_datasets`)


In [ ]:
from pyterrier.datasets import get_dataset

msmarco = get_dataset("irds:msmarco-passage")
msmarco_trec_2019 = get_dataset("irds:msmarco-passage/trec-dl-2019/judged")
msmarco_trec_2020 = get_dataset("irds:msmarco-passage/trec-dl-2020/judged")

2. Index MSMARCO passages (skip if index already exists)


In [ ]:
from os.path import exists
from pyterrier import IndexRef
from pyterrier.index import IterDictIndexer

if not exists("./msmarco-index/"):
    index = IterDictIndexer("./msmarco-index/").index(msmarco.get_corpus_iter())
else:
    index = IndexRef.of("./msmarco-index/")

### Baseline Rankers

- define initial retrieval with [PyTerrier](https://pyterrier.readthedocs.io/)
  - BM25
  - PL2


In [ ]:
from pyterrier.batchretrieve import BatchRetrieve

bm25 = BatchRetrieve(index, wmodel="BM25")
pl2 = BatchRetrieve(index, wmodel="PL2")

## Post-hoc Axiomatic Analyses

- create axiomatic analyses with the `AxiomaticExperiment` class
- parameters:
  - retrieval systems
  - axioms
  - depth (how many documents to consider when building preference matrices)
  - topics and qrels
  - index location
  - document dataset (for full texts)


In [ ]:
from ir_axioms.axiom import TFC1, STMC1_fastText, STMC2_fastText, LB1
from ir_axioms.backend.pyterrier.experiment import AxiomaticExperiment

experiment = AxiomaticExperiment(
    [bm25, pl2],
    names=["BM25", "PL2"],
    axioms=[TFC1(), STMC1_fastText(), STMC2_fastText(), LB1()],
    depth=5,
    topics=msmarco_trec_2020.get_topics(),
    qrels=msmarco_trec_2020.get_qrels(),
    index=index,
    dataset=msmarco,
    filter_by_qrels=True,
    verbose=True,
)

### Calculate Pairwise Preferences

- `DataFrame` of all pairwise preferences (up to the specified depth)
- rows: document pairs
- colums: preferences of all axioms + original DF columns
- used in further analyses


In [ ]:
experiment.preferences

In [ ]:
experiment.preferences[
    [
        "name",
        "query",
        "docno_a",
        "docno_b",
        "ORIG_preference",
        "ORACLE_preference",
        "STMC1-fastText_preference",
    ]
].head()

### Preference Distribution

- distribution of axiom preferences compared to ORIG
- how many...
  - return a "zero" preference?
  - agree with ORIG?
  - disagree with ORIG?
- identify "useful" axioms
- interesting: axioms that often disagree with ORIG


In [ ]:
experiment.preference_distribution

### Inconsistent Pairs

- document pairs where...
  - ORIG contradicts ORACLE
  - at least 1 axiom could correct the mistake
- useful for focussed debugging of retrieval systems


In [ ]:
inconsistent_pair = experiment.inconsistent_pairs.head(1)
inconsistent_pair[
    [
        "name",
        "query",
        "docno_a",
        "docno_b",
        "ORIG_preference",
        "ORACLE_preference",
        "STMC1-fastText_preference",
    ]
]

In [ ]:
msmarco.irds_ref().docs_store().get(inconsistent_pair["docno_a"].iloc[0]).text

## Axiomatic Re-Ranking

- improve consistency with axioms
- steps:
  1. combine/weight multiple axioms
  1. re-rank with combined axiom


### Majority Vote

- idea: only change order where multiple axioms agree
  - majority vote between specified axioms
  - each axiom votes for its preference
  - fall back to original ranking order if not a majority >= 50%
- combine axioms with Python operators
  `%`: majority vote (with 50% majority)
  `|`: fallback, if preference is 0
- combined axiom is itself an `Axiom`


In [ ]:
from ir_axioms.axiom import ORIG

majority_vote = (TFC1() % STMC1_fastText() % STMC2_fastText() % LB1()) | ORIG()

### KwikSort

- similar to quicksort
  1. select pivot
  1. compare and split rankings before and after the pivot
  1. repeat for both sub-rankings
- results depend on pivot selection
- implemented as PyTerrier `Transformer`
  - easy to use with PyTerrier pipelines (e.g., to re-rank top-10)


In [ ]:
from ir_axioms.backend.pyterrier.transformers import KwikSortReranker

kwiksort = KwikSortReranker(
    axiom=majority_vote, index=index, dataset=msmarco, verbose=True
)
bm25_kwiksort = (bm25 % 10 >> kwiksort) ^ bm25
pl2_kwiksort = (pl2 % 10 >> kwiksort) ^ pl2

### Evaluate Re-Rankers

- evaluate with PyTerrier `Experiment`
- use re-ranked pipelines like any other PyTerrier pipeline


In [ ]:
from pyterrier.pipelines import Experiment

experiment = Experiment(
    [bm25, bm25_kwiksort, pl2, pl2_kwiksort],
    names=["BM25", "BM25 + KwikSort", "PL2", "PL2 + KwikSort"],
    topics=msmarco_trec_2020.get_topics(),
    qrels=msmarco_trec_2020.get_qrels(),
    eval_metrics=["ndcg_cut_10", "map"],
    verbose=True,
)
experiment

#### Results

- only slight changes in effectiveness
- but improved consistency with axioms
- more explainable
